In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [43]:
%cd /content/PatternSmith
!python -m pip install -e .

/content/PatternSmith
Obtaining file:///content/PatternSmith
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for patternsmith (pyproject.toml) ... done
  Created wheel for patternsmith: filename=patternsmith-0.1.0-0.editable-py3-none-any.whl size=1293 sha256=0a3588f6d5e430fdad792002450ce8804b4b7b96adc5c8ca39d784996bfde6b5
  Stored in directory: /tmp/pip-ephem-wheel-cache-2u314mt_/wheels/63/f7/da/0803404cc1001087c2b63edce523ad64e6e5bd57baa2307e45
Successfully built patternsmith
  Attempting uninstall: patternsmith
    Found existing installation: patternsmith 0.1.0
    Uninstalling patternsmith-0.1.0:
      Successfully uninstalled patternsmith-0.1.0


In [45]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU unavailable. Check Colab runtime settings.")


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [46]:

from pathlib import Path

data_dir = Path("/content/PatternSmith/data/splits/NL-RX-Turk")

for split in ["train", "validation", "test"]:
    src = data_dir / split / "src.txt"
    targ = data_dir / split / "targ.txt"

    assert src.exists(), f"Missing {src}"
    assert targ.exists(), f"Missing {targ}"

    source_count = len(src.read_text(encoding="utf-8").splitlines())
    target_count = len(targ.read_text(encoding="utf-8").splitlines())

    print(f"{split}: {source_count} descriptions, {target_count} regexes")
    assert source_count == target_count


train: 6245 descriptions, 6245 regexes
validation: 961 descriptions, 961 regexes
test: 2402 descriptions, 2402 regexes


**Smoketest**

In [48]:
%cd /content/PatternSmith

!python scripts/train.py \
    --device cuda \
    --epochs 2 \
    --batch-size 4 \
    --output-dir /content/drive/MyDrive/PatternSmith/checkpoints_smoke

/content/PatternSmith
PatternSmith Training
Device: cuda
Dataset directory: /content/PatternSmith/data/splits/NL-RX-Turk
Output directory: /content/drive/MyDrive/PatternSmith/checkpoints_smoke
GPU: Tesla T4

Training examples: 6245
Validation examples: 961
Training batches per epoch: 1562
Validation batches per epoch: 241

Trainable parameters: 7,242,624
Total optimizer steps: 3124
Warmup steps: 156

Epoch 1/2
-----------------------------------------------------------------
Train | Loss: 1.8654 | Perplexity: 6.4584
Valid | Loss: 0.5311 | Perplexity: 1.7008
Learning rate: 0.00017613
Mean gradient norm: 3.3836
Training steps: 1562 | Supervised tokens: 114957
New best validation loss: 0.5311
Early stopping counter: 0/5

Epoch 2/2
-----------------------------------------------------------------
Train | Loss: 0.4379 | Perplexity: 1.5495
Valid | Loss: 0.3624 | Perplexity: 1.4367
Learning rate: 0.00003000
Mean gradient norm: 4.3316
Training steps: 1562 | Supervised tokens: 114957
New best v

**Full Training**

In [50]:
%cd /content/PatternSmith

!python scripts/train.py \
    --device cuda \
    --epochs 40 \
    --batch-size 16 \
    --output-dir /content/drive/MyDrive/PatternSmith/checkpoints_full

/content/PatternSmith
PatternSmith Training
Device: cuda
Dataset directory: /content/PatternSmith/data/splits/NL-RX-Turk
Output directory: /content/drive/MyDrive/PatternSmith/checkpoints_full
GPU: Tesla T4

Training examples: 6245
Validation examples: 961
Training batches per epoch: 391
Validation batches per epoch: 61

Trainable parameters: 7,242,624
Total optimizer steps: 15640
Warmup steps: 782

Epoch 1/40
-----------------------------------------------------------------
Train | Loss: 7.9182 | Perplexity: 2746.8540
Valid | Loss: 3.4213 | Perplexity: 30.6083
Learning rate: 0.00015038
Mean gradient norm: 2.2015
Training steps: 391 | Supervised tokens: 114957
New best validation loss: 3.4213
Early stopping counter: 0/5

Epoch 2/40
-----------------------------------------------------------------
Train | Loss: 1.7382 | Perplexity: 5.6869
Valid | Loss: 0.8468 | Perplexity: 2.3322
Learning rate: 0.00030000
Mean gradient norm: 2.3011
Training steps: 391 | Supervised tokens: 114957
New best

resume from the latest completed epoch

In [51]:
%cd /content/PatternSmith

!python scripts/train.py \
    --device cuda \
    --epochs 40 \
    --batch-size 16 \
    --output-dir /content/drive/MyDrive/PatternSmith/checkpoints_full \
    --resume /content/drive/MyDrive/PatternSmith/checkpoints_full/last.pt

/content/PatternSmith
PatternSmith Training
Device: cuda
Dataset directory: /content/PatternSmith/data/splits/NL-RX-Turk
Output directory: /content/drive/MyDrive/PatternSmith/checkpoints_full
GPU: Tesla T4

Training examples: 6245
Validation examples: 961
Training batches per epoch: 391
Validation batches per epoch: 61

Trainable parameters: 7,242,624
Total optimizer steps: 15640
Warmup steps: 782

Resuming from: /content/drive/MyDrive/PatternSmith/checkpoints_full/last.pt
Resuming from epoch 22
Best validation loss so far: 0.2044

Epoch 22/40
-----------------------------------------------------------------
Train | Loss: 0.0893 | Perplexity: 1.0934
Valid | Loss: 0.2216 | Perplexity: 1.2481
Learning rate: 0.00015385
Mean gradient norm: 1.9538
Training steps: 391 | Supervised tokens: 114957
Early stopping counter: 6/5

Early stopping triggered.

Best model restored.
Best epoch: 16
Best validation loss: 0.2044

Training completed.
Latest checkpoint: /content/drive/MyDrive/PatternSmith/ch

In [52]:

import torch

checkpoint_path = (
    "/content/drive/MyDrive/PatternSmith/checkpoints_full/best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

print("Checkpoint keys:")
for key, value in checkpoint.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for subkey in value:
            print("  ", subkey)
    else:
        print(f"{key}: {value}")

print("\nModel configuration:")
print(checkpoint.get("config", {}).get("model"))


Checkpoint keys:
epoch: 16

model_state_dict:
   embedding.token_embedding.weight
   embedding.position_embedding.weight
   transformer_blocks.0.norm1.weight
   transformer_blocks.0.norm1.bias
   transformer_blocks.0.attention.query_projection.weight
   transformer_blocks.0.attention.query_projection.bias
   transformer_blocks.0.attention.key_projection.weight
   transformer_blocks.0.attention.key_projection.bias
   transformer_blocks.0.attention.value_projection.weight
   transformer_blocks.0.attention.value_projection.bias
   transformer_blocks.0.attention.output_projection.weight
   transformer_blocks.0.attention.output_projection.bias
   transformer_blocks.0.norm2.weight
   transformer_blocks.0.norm2.bias
   transformer_blocks.0.mlp.fc1.weight
   transformer_blocks.0.mlp.fc1.bias
   transformer_blocks.0.mlp.fc2.weight
   transformer_blocks.0.mlp.fc2.bias
   transformer_blocks.1.norm1.weight
   transformer_blocks.1.norm1.bias
   transformer_blocks.1.attention.query_projection.weight

**create a zip and move to vs code**

In [57]:

from pathlib import Path
import shutil
import zipfile
import os

project_dir = Path("/content/PatternSmith")

checkpoint_source = Path(
    "/content/drive/MyDrive/PatternSmith/checkpoints_full/best.pt"
)

# Ensure the trained checkpoint is included.
checkpoint_dir = project_dir / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

shutil.copy2(
    checkpoint_source,
    checkpoint_dir / "best.pt",
)

# Create ZIP locally first for faster compression.
local_zip = Path("/content/PatternSmith_final.zip")

excluded = {
    ".git",
    ".venv",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
}

with zipfile.ZipFile(
    local_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
) as archive:

    for file in project_dir.rglob("*"):
        if not file.is_file():
            continue

        relative = file.relative_to(project_dir)

        if any(part in excluded for part in relative.parts):
            continue

        if file.suffix == ".pyc":
            continue

        archive.write(
            file,
            Path("PatternSmith") / relative,
        )

print("ZIP created:", local_zip)
print("Size:", round(local_zip.stat().st_size / (1024 * 1024), 2), "MB")

# Copy ZIP into Google Drive.
drive_zip = Path(
    "/content/drive/MyDrive/PatternSmith_final.zip"
)

shutil.copy2(local_zip, drive_zip)

print("Successfully copied to Google Drive!")
print("Drive path:", drive_zip)


ZIP created: /content/PatternSmith_final.zip
Size: 75.64 MB
Successfully copied to Google Drive!
Drive path: /content/drive/MyDrive/PatternSmith_final.zip
